In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd  

In [2]:
df = pd.read_csv('diabetes.csv')

In [3]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [4]:
df.corr()['Outcome'].sort_values(ascending=False)

Outcome                     1.000000
Glucose                     0.466581
BMI                         0.292695
Age                         0.238356
Pregnancies                 0.221898
DiabetesPedigreeFunction    0.173844
Insulin                     0.130548
SkinThickness               0.074752
BloodPressure               0.065068
Name: Outcome, dtype: float64

In [5]:
X = df.iloc[:,0:-1].values
y = df.iloc[:,-1].values

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X = scaler.fit_transform(X)


In [7]:
X

array([[ 0.63994726,  0.84832379,  0.14964075, ...,  0.20401277,
         0.46849198,  1.4259954 ],
       [-0.84488505, -1.12339636, -0.16054575, ..., -0.68442195,
        -0.36506078, -0.19067191],
       [ 1.23388019,  1.94372388, -0.26394125, ..., -1.10325546,
         0.60439732, -0.10558415],
       ...,
       [ 0.3429808 ,  0.00330087,  0.14964075, ..., -0.73518964,
        -0.68519336, -0.27575966],
       [-0.84488505,  0.1597866 , -0.47073225, ..., -0.24020459,
        -0.37110101,  1.17073215],
       [-0.84488505, -0.8730192 ,  0.04624525, ..., -0.20212881,
        -0.47378505, -0.87137393]], shape=(768, 8))

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [9]:
import tensorflow 
from tensorflow import keras
from keras import Sequential
from keras.layers import Dense,Dropout


c:\Program Files\Python313\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [10]:
model = Sequential()
model.add(Dense(32, input_dim=8, activation="relu"))
model.add(Dense(1, activation="sigmoid"))
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

c:\Program Files\Python313\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test))

Epoch 1/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 23ms/step - accuracy: 0.6824 - loss: 0.6381 - val_accuracy: 0.6818 - val_loss: 0.6162
Epoch 2/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7101 - loss: 0.6007 - val_accuracy: 0.7273 - val_loss: 0.5826
Epoch 3/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7166 - loss: 0.5734 - val_accuracy: 0.7403 - val_loss: 0.5588
Epoch 4/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7280 - loss: 0.5530 - val_accuracy: 0.7403 - val_loss: 0.5406
Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7410 - loss: 0.5383 - val_accuracy: 0.7468 - val_loss: 0.5289
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7410 - loss: 0.5263 - val_accuracy: 0.7403 - val_loss: 0.5196
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7573 - loss: 0.5149 - val_accuracy: 0.7532 - val_loss: 0.5083
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7622 - loss: 0.5060 - val_accuracy: 0.7662 -

In [12]:
# how to select appropriate optimizer
# No of nodes in layer
# No of layers
# all in one

In [13]:
import kerastuner as kt

C:\Users\HP\AppData\Local\Temp\ipykernel_5400\1654478174.py:1: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  import kerastuner as kt


In [14]:
def build_model(hp):

    model = Sequential()
    model.add(Dense(32,activation='relu',input_dim=8))
    model.add(Dense(1,activation='sigmoid'))

    optimizer = hp.Choice('optimizer',values=['adam','sgd','rmsprop','adadelta'])
    model.compile(loss='binary_crossentropy',optimizer=optimizer,metrics=['accuracy'])

    return model

In [15]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5)

Reloading Tuner from .\untitled_project\tuner0.json


In [16]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [17]:
tuner.get_best_hyperparameters()[0].values

{'optimizer': 'sgd'}

In [18]:
model = tuner.get_best_models(num_models=1)[0]

In [19]:
def build_model(hp):
    model = Sequential()
    
    units = hp.Int('units', min_value=8, max_value=128)

    model.add(Dense(units=units, activation='relu', input_dim=8))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer='rmsprop', metrics=['accuracy'])

    return model


In [20]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='aa')

Reloading Tuner from aa\untitled_project\tuner0.json


In [21]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [22]:
tuner.get_best_hyperparameters()[0].values

{'units': 101}

In [23]:
model = tuner.get_best_models(num_models=1)[0]

c:\Program Files\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 6 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [24]:
model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2)

Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.7699 - loss: 0.4785 - val_accuracy: 0.7724 - val_loss: 0.4439
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7760 - loss: 0.4696 - val_accuracy: 0.7642 - val_loss: 0.4435
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7739 - loss: 0.4651 - val_accuracy: 0.7724 - val_loss: 0.4428
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7800 - loss: 0.4614 - val_accuracy: 0.7724 - val_loss: 0.4415
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.7739 - loss: 0.4592 - val_accuracy: 0.7724 - val_loss: 0.4423
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7821 - loss: 0.4562 - val_accuracy: 0.7805 - val_loss: 0.4417
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7719 - loss: 0.4546 - val_accuracy: 0.7805 - val_loss: 0.4425
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7780 - loss: 0.4528 - val_accuracy: 0.7805 

In [25]:
def build_model(hp):
    model = Sequential()
    model.add(Dense(111,activation='relu',input_dim=8))

    for i in range(hp.Int('num_layers', 1, 10)):
        model.add(Dense(111,activation='relu'))
    
    model.add(Dense(1,activation='sigmoid'))
    model.compile(loss='binary_crossentropy',optimizer='rmsprop',metrics=['accuracy'])
    return model


In [26]:
tuner = kt.RandomSearch(build_model,objective='val_accuracy',max_trials=5,directory='bb')

Reloading Tuner from bb\untitled_project\tuner0.json


In [27]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

In [28]:
tuner.get_best_hyperparameters()[0].values

{'num_layers': 7}

In [29]:
model = tuner.get_best_models(num_models=1)[0]

c:\Program Files\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 20 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [30]:
model.fit(X_train, y_train, epochs=100,initial_epoch=4, validation_data=(X_test, y_test))

Epoch 5/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 3s 23ms/step - accuracy: 0.7801 - loss: 0.4761 - val_accuracy: 0.7922 - val_loss: 0.4935
Epoch 6/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.7948 - loss: 0.4458 - val_accuracy: 0.7468 - val_loss: 0.5640
Epoch 7/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.7720 - loss: 0.4494 - val_accuracy: 0.7597 - val_loss: 0.5024
Epoch 8/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8013 - loss: 0.4281 - val_accuracy: 0.8052 - val_loss: 0.5011
Epoch 9/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8094 - loss: 0.4217 - val_accuracy: 0.7987 - val_loss: 0.5031
Epoch 10/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.8111 - loss: 0.4153 - val_accuracy: 0.7987 - val_loss: 0.5019
Epoch 11/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.8355 - loss: 0.3900 - val_accuracy: 0.7727 - val_loss: 0.4876
Epoch 12/100
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8339 - loss: 0.3852 - val_accuracy: 0.77

In [31]:
def build_model(hp):
    model = Sequential()
    units = hp.Int('units', 1, 10)
    counter = 0
    for i in range(hp.Int('num_layers', 1, 10)):
        if counter == 0:
            model.add(Dense(hp.Int('units'+str(i), min_value=8, max_value=128,step=8),
             activation=hp.Choice('activation'+str(i), ['relu', 'tanh', 'sigmoid']),
              input_dim=8))
            model.add(Dropout(hp.Float('dropout'+str(i), min_value=0.1, max_value=0.9, step=0.1)))
        else:
            model.add(Dense(hp.Int('units'+str(i), min_value=8, max_value=128,step=8),
             activation=hp.Choice('activation'+str(i), ['relu', 'tanh', 'sigmoid']))) 
            model.add(Dropout(hp.Float('dropout'+str(i), min_value=0.1, max_value=0.9, step=0.1)))
        counter += 1
    model.add(Dense(1, activation='sigmoid'))

    model.compile(loss='binary_crossentropy', optimizer=hp.Choice('optimizer', ['adam', 'sgd','rmsprop','adagrad']), metrics=['accuracy'])

    return model

In [32]:
tuner = kt.RandomSearch(build_model, objective="val_accuracy", max_trials=3,directory="final1")


In [33]:
tuner.search(X_train,y_train,epochs=5,validation_data=(X_test,y_test))

Trial 3 Complete [00h 00m 03s]
val_accuracy: 0.7077922224998474

Best val_accuracy So Far: 0.7662337422370911
Total elapsed time: 00h 00m 09s


In [34]:
tuner.get_best_hyperparameters()[0].values

{'units': 10,
 'num_layers': 3,
 'units0': 104,
 'activation0': 'sigmoid',
 'dropout0': 0.1,
 'optimizer': 'rmsprop',
 'units1': 8,
 'activation1': 'relu',
 'dropout1': 0.1,
 'units2': 8,
 'activation2': 'relu',
 'dropout2': 0.1}

In [35]:
model = tuner.get_best_models(num_models=1)[0]

c:\Program Files\Python313\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 2 variables whereas the saved optimizer has 10 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [36]:
model.fit(X_train, y_train, epochs=1000,initial_epoch=6,validation_data=(X_test, y_test))

Epoch 7/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step - accuracy: 0.7166 - loss: 0.5569 - val_accuracy: 0.7597 - val_loss: 0.5211
Epoch 8/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7020 - loss: 0.5424 - val_accuracy: 0.7792 - val_loss: 0.5233
Epoch 9/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7215 - loss: 0.5348 - val_accuracy: 0.7727 - val_loss: 0.5042
Epoch 10/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7606 - loss: 0.5238 - val_accuracy: 0.7987 - val_loss: 0.5182
Epoch 11/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7378 - loss: 0.5328 - val_accuracy: 0.7792 - val_loss: 0.4934
Epoch 12/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7476 - loss: 0.5297 - val_accuracy: 0.7922 - val_loss: 0.4902
Epoch 13/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7280 - loss: 0.5261 - val_accuracy: 0.8052 - val_loss: 0.5080
Epoch 14/1000
20/20 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.7394 - loss: 0.5147 - val_accura